In [0]:
%sql

CREATE OR REFRESH STREAMING TABLE bridge_monitoring.`03_gold`.bridge_metrics
  COMMENT "10-min avg temperature, max vibration & max tilt per bridge with window start/end"
AS
WITH temp_agg AS (
  SELECT
    bridge_id,
    bridge_name,
    country,
    location,
    window.start AS window_start,
    window.end   AS window_end,
    AVG(temperature) AS avg_temperature
  FROM STREAM(bridge_monitoring.`02_silver`.bridge_temperature)
       WATERMARK event_time DELAY OF INTERVAL 2 MINUTES
  GROUP BY window(event_time, '10 minutes'),
           bridge_id, bridge_name, country, location
),
vib_agg AS (
  SELECT
    bridge_id,
    window.start AS window_start,
    window.end   AS window_end,
    MAX(vibration) AS max_vibration
  FROM STREAM(bridge_monitoring.`02_silver`.bridge_vibration)
       WATERMARK event_time DELAY OF INTERVAL 2 MINUTES
  GROUP BY window(event_time, '10 minutes'), bridge_id
),
tilt_agg AS (
  SELECT
    bridge_id,
    window.start AS window_start,
    window.end   AS window_end,
    MAX(tilt_angle) AS max_tilt_angle
  FROM STREAM(bridge_monitoring.`02_silver`.bridge_tilt)
       WATERMARK event_time DELAY OF INTERVAL 2 MINUTES
  GROUP BY window(event_time, '10 minutes'), bridge_id
)
SELECT
  t.bridge_id,
  t.bridge_name,
  t.country,
  t.location,
  t.window_start,
  t.window_end,
  ROUND(t.avg_temperature, 2) AS avg_temperature,
  v.max_vibration,
  w.max_tilt_angle
FROM STREAM(temp_agg) t
JOIN vib_agg v
  ON  t.bridge_id    = v.bridge_id
  AND t.window_start = v.window_start
  AND t.window_end   = v.window_end
JOIN tilt_agg w
  ON  t.bridge_id    = w.bridge_id
  AND t.window_start = w.window_start
  AND t.window_end   = w.window_end;

--Here are the DataFrames in the session:
--- Spark DataFrame

In [0]:
%sql
SELECT * FROM bridge_monitoring.`03_gold`.bridge_metrics

In [0]:
# import dlt
# from pyspark.sql.functions import col, window, max, avg, round

# @dlt.table(
#     name="bridge_monitoring.03_gold.bridge_metrics",
#     comment="10-min avg temperature, max vibration & max tilt per bridge with window start/end"
# )
# def bridge_metrics():
#     temp = (
#         dlt.read_stream("bridge_monitoring.02_silver.bridge_temperature")
#         .withWatermark("event_time", "2 minutes")
#     )
#     vib = (
#         dlt.read_stream("bridge_monitoring.02_silver.bridge_vibration")
#         .withWatermark("event_time", "2 minutes")
#     )
#     tilt = (
#         dlt.read_stream("bridge_monitoring.02_silver.bridge_tilt")
#         .withWatermark("event_time", "2 minutes")
#     )
    
#     temp_agg = (
#         temp.groupBy(
#             window("event_time", "10 minutes"),
#             col("bridge_id"),
#             col("bridge_name"),
#             col("country"),
#             col("location")
#         )
#         .agg(
#             avg("temperature").alias("avg_temperature")
#         )
#         .select(
#             col("bridge_id"),
#             col("bridge_name"),
#             col("country"),
#             col("location"),
#             col("window.start").alias("window_start"),
#             col("window.end").alias("window_end"),
#             col("avg_temperature")
#         )
#     )
    
#     vib_agg = (
#         vib.groupBy(
#             window("event_time", "10 minutes"),
#             col("bridge_id"),
#         )
#         .agg(
#             max("vibration").alias("max_vibration")
#         )
#         .select(
#             col("bridge_id"),
#             col("window.start").alias("window_start"),
#             col("window.end").alias("window_end"),
#             col("max_vibration")
#         )
#     )
    
#     tilt_agg = (
#         tilt.groupBy(
#             window("event_time", "10 minutes"),
#             col("bridge_id"),
#             )
#         .agg(
#             max("tilt_angle").alias("max_tilt_angle")
#         )
#         .select(
#             col("bridge_id"),
#             col("window.start").alias("window_start"),
#             col("window.end").alias("window_end"),
#             col("max_tilt_angle")
#         )
#     )

#     return (temp_agg.alias("t")
#             .join(
#                 vib_agg.alias("v"), 
#                 on=["bridge_id", "window_start", "window_end"],
#                 how="inner")
#             .join(tilt_agg.alias("w"), 
#                 on=["bridge_id", "window_start", "window_end"],
#                 how="inner")
#             .select(
#                 col("bridge_id"),
#                 col("brdige_name"),
#                 col("country"),
#                 col("location"),
#                 col("window_start"),
#                 col("window_end"),
#                 col(round("avg_temperature", 2)),
#                 col("max_vibration"),
#                 col("max_tilt_angle")
#             )
#     )
            